In [1]:
!pip install biopython

In [2]:
from Bio.PDB import PDBList, PDBParser
import numpy as np

In [3]:
from Bio.PDB import PDBList

pdbl = PDBList()

file_path = pdbl.retrieve_pdb_file(
    "2V8T",
    pdir=".",
    file_format="pdb"
)

print(file_path)

.\pdb2v8t.ent


In [4]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)

structure = parser.get_structure(
    "2V8T",
    file_path
)

model = structure[0]

print("Structure loaded successfully")

Structure loaded successfully


In [5]:
for chain in model:
    print("Chain:", chain.id)

Chain: A
Chain: B


In [6]:
chain = model["A"]

for residue in chain:
    res_id = residue.id[1]

    if 200 <= res_id <= 215:
        print(res_id, residue.resname)

200 THR
201 GLY
202 VAL
203 GLU
204 MET
205 THR
206 LYS
207 MET
208 LEU
209 PRO
210 ILE
211 PRO
212 LYS
213 ILE
214 ASP
215 ASN


In [7]:
three_to_one = {
    "ALA":"A", "ARG":"R", "ASN":"N", "ASP":"D",
    "CYS":"C", "GLN":"Q", "GLU":"E", "GLY":"G",
    "HIS":"H", "ILE":"I", "LEU":"L", "LYS":"K",
    "MET":"M", "PHE":"F", "PRO":"P", "SER":"S",
    "THR":"T", "TRP":"W", "TYR":"Y", "VAL":"V"
}

sequence = ""

for residue in chain:
    res_id = residue.id[1]

    if 200 <= res_id <= 215:
        sequence += three_to_one[residue.resname]

print("Sequence:", sequence)

Sequence: TGVEMTKMLPIPKIDN


In [8]:
region_atoms = []

for residue in chain:
    res_id = residue.id[1]

    if 200 <= res_id <= 215:
        for atom in residue:
            region_atoms.append(atom)

total_mass = 0
weighted_coordinates = np.zeros(3)

for atom in region_atoms:
    mass = atom.mass
    coord = atom.coord

    total_mass += mass
    weighted_coordinates += mass * coord

center_of_mass = weighted_coordinates / total_mass

print("Center of mass:")
print("X =", center_of_mass[0])
print("Y =", center_of_mass[1])
print("Z =", center_of_mass[2])

Center of mass:
X = -7.0336260799144386
Y = 17.619905999790706
Z = 40.67338677641589


In [9]:
def calculate_center_of_mass(chain, start, end):

    total_mass = 0.0
    weighted_coordinates = np.zeros(3)

    for residue in chain:

        res_id = residue.id[1]

        if start <= res_id <= end:

            for atom in residue:

                mass = atom.mass
                coordinate = atom.coord

                total_mass += mass
                weighted_coordinates += mass * coordinate

    center = weighted_coordinates / total_mass

    return center


com = calculate_center_of_mass(
    chain,
    200,
    215
)

print("Center of Mass of residues 200–215:")
print(f"X = {com[0]:.3f} Å")
print(f"Y = {com[1]:.3f} Å")
print(f"Z = {com[2]:.3f} Å")

Center of Mass of residues 200–215:
X = -7.034 Å
Y = 17.620 Å
Z = 40.673 Å


In [10]:
from Bio.PDB import NeighborSearch

# Collect atoms from the region 200–215
region_atoms = []

for residue in chain:
    if 200 <= residue.id[1] <= 215:
        region_atoms.extend(list(residue.get_atoms()))

# Find all non-protein atoms
ligand_atoms = []

for ch in model:
    for residue in ch:
        if residue.id[0] != " ":
            ligand_atoms.extend(list(residue.get_atoms()))

# Calculate minimum distance
minimum_distance = float("inf")

for atom1 in region_atoms:
    for atom2 in ligand_atoms:

        distance = np.linalg.norm(
            atom1.coord - atom2.coord
        )

        if distance < minimum_distance:
            minimum_distance = distance
            closest_atom1 = atom1
            closest_atom2 = atom2

print("Shortest distance =", minimum_distance, "Å")
print("Region atom:", closest_atom1.get_name())
print("Other atom:", closest_atom2.get_name())

Shortest distance = 2.4794624 Å
Region atom: NZ
Other atom: O
